### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ionosphere",
    dataset_year="1988",
    domain_str="physics & astronomy",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5W01B",
    download_description="""

mkdir -p local-data-warehouse/ionosphere/ && wget -P local-data-warehouse/ionosphere/ https://archive.ics.uci.edu/static/public/52/ionosphere.zip && unzip local-data-warehouse/ionosphere/ionosphere.zip -d local-data-warehouse/ionosphere/ && rm local-data-warehouse/ionosphere/ionosphere.zip
""",
    # References
    academic_reference_bibtex="""@inproceedings{Sigillito1989ClassificationOR,
  title={Classification of radar returns from the ionosphere using neural networks},
  author={Vincent G. Sigillito and Simon Wing and Larrie V. Hutton and K. L. Baker},
  year={1989},
  url={https://api.semanticscholar.org/CorpusID:18522381}
}
""",
    academic_reference_bibtex_key="Sigillito1989ClassificationOR",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We drop "Attribute 2" as it has constant value across samples.
- There is one row duplicate that we keep.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="GoodSignal",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="GoodSignal",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "ionosphere.data", header=None)

target_feature = "GoodSignal"
df.columns = [f"Attribute {i+1}" for i in range(len(df.columns) - 1)] + [target_feature]
df[target_feature] = df[target_feature].map({"g": "Yes", "b": "No"}).astype("category")

df = df.drop(["Attribute 2"], axis=1)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 351
Columns: 34
Use sampling: False (sample size: 351)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Attribute 28', 'Attribute 18', 'Attribute 26', 'Attribute 16', 'Attribute 12', 'Attribute 4', 'Attribute 10', 'Attribute 14', 'Attribute 30', 'Attribute 20']
Rows remaining as candidates after top-10 filter: 2 (of 351)

#### Duplicate Report
Total duplicate rows: 1 (0.28% of dataset)
Duplicate rows ignoring target: 1 (0.28% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Attribute 1,Attribute 3,Attribute 4,Attribute 5,Attribute 6,Attribute 7,Attribute 8,Attribute 9,Attribute 10,Attribute 11,Attribute 12,Attribute 13,Attribute 14,Attribute 15,Attribute 16,Attribute 17,Attribute 18,Attribute 19,Attribute 20,Attribute 21,Attribute 22,Attribute 23,Attribute 24,Attribute 25,Attribute 26,Attribute 27,Attribute 28,Attribute 29,Attribute 30,Attribute 31,Attribute 32,Attribute 33,Attribute 34,GoodSignal
0,1,0.99539,-0.05889,0.85243,0.02306,0.83398,-0.37708,1.00000,0.03760,0.85243,-0.17755,0.59755,-0.44945,0.60536,-0.38223,0.84356,-0.38542,0.58212,-0.32192,0.56971,-0.29674,0.36946,-0.47357,0.56811,-0.51171,0.41078,-0.46168,0.21266,-0.34090,0.42267,-0.54487,0.18641,-0.45300,Yes
1,1,1.00000,-0.18829,0.93035,-0.36156,-0.10868,-0.93597,1.00000,-0.04549,0.50874,-0.67743,0.34432,-0.69707,-0.51685,-0.97515,0.05499,-0.62237,0.33109,-1.00000,-0.13151,-0.45300,-0.18056,-0.35734,-0.20332,-0.26569,-0.20468,-0.18401,-0.19040,-0.11593,-0.16626,-0.06288,-0.13738,-0.02447,No
2,1,1.00000,-0.03365,1.00000,0.00485,1.00000,-0.12062,0.88965,0.01198,0.73082,0.05346,0.85443,0.00827,0.54591,0.00299,0.83775,-0.13644,0.75535,-0.08540,0.70887,-0.27502,0.43385,-0.12062,0.57528,-0.40220,0.58984,-0.22145,0.43100,-0.17365,0.60436,-0.24180,0.56045,-0.38238,Yes
3,1,1.00000,-0.45161,1.00000,1.00000,0.71216,-1.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,-1.00000,0.14516,0.54094,-0.39330,-1.00000,-0.54467,-0.69975,1.00000,0.00000,0.00000,1.00000,0.90695,0.51613,1.00000,1.00000,-0.20099,0.25682,1.00000,-0.32382,1.00000,No
4,1,1.00000,-0.02401,0.94140,0.06531,0.92106,-0.23255,0.77152,-0.16399,0.52798,-0.20275,0.56409,-0.00712,0.34395,-0.27457,0.52940,-0.21780,0.45107,-0.17813,0.05982,-0.35575,0.02309,-0.52879,0.03286,-0.65158,0.13290,-0.53206,0.02431,-0.62197,-0.05707,-0.59573,-0.04608,-0.65697,Yes


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,GoodSignal,category,0.0,0.0,2.0,"Yes, No"
1,Attribute 3,float64,0.0,0.0,219.0,"1.0, 0.0, -1.0, 0.9954, 0.6068, 0.8941, 0.9463, 0.9177, 0.7663, 0.5894"
2,Attribute 4,float64,0.0,0.0,269.0,"0.0, 1.0, -1.0, 0.0008, -0.5, 0.0909, -0.0952, 0.4546, 0.3912, -0.0349"
3,Attribute 5,float64,0.0,0.0,204.0,"1.0, 0.0, -1.0, 0.8524, 0.9011, 0.8056, 0.9677, 0.4278, 0.7653, 0.6112"
4,Attribute 6,float64,0.0,0.0,259.0,"0.0, 1.0, -1.0, 0.0231, 0.251, -0.1214, 0.553, 0.3811, 0.3554, 0.3314"
5,Attribute 7,float64,0.0,0.0,231.0,"1.0, 0.0, -1.0, 0.5, 0.3333, 0.834, 0.8159, 0.701, 0.9668, 0.7703"
6,Attribute 8,float64,0.0,0.0,260.0,"0.0, 1.0, -1.0, 0.0909, 0.0687, -0.3771, 0.4575, 0.4996, 0.4339, 0.3304"
7,Attribute 9,float64,0.0,0.0,244.0,"1.0, 0.0, -1.0, 0.8, -0.3333, 0.6364, -0.0364, 0.732, 0.4696, 0.6932"
8,Attribute 10,float64,0.0,0.0,267.0,"0.0, 1.0, -1.0, -0.1429, -0.0909, -0.0751, 0.6384, 0.5998, 0.5302, 0.4125"
9,Attribute 11,float64,0.0,0.0,246.0,"1.0, 0.0, -1.0, 0.5, 0.5617, 0.3278, 0.1375, 0.5038, 0.5921, 0.54"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Attribute 1,351.0,0.891738,0.311155,0.0,1.0
Attribute 3,351.0,0.641342,0.497708,-1.0,1.0
Attribute 4,351.0,0.044372,0.441435,-1.0,1.0
Attribute 5,351.0,0.601068,0.519862,-1.0,1.0
Attribute 6,351.0,0.115889,0.460810,-1.0,1.0
Attribute 7,351.0,0.550095,0.492654,-1.0,1.0
Attribute 8,351.0,0.119360,0.520750,-1.0,1.0
Attribute 9,351.0,0.511848,0.507066,-1.0,1.0
Attribute 10,351.0,0.181345,0.483851,-1.0,1.0
Attribute 11,351.0,0.476183,0.563496,-1.0,1.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count   pct
column     rank                   
GoodSignal 1      Yes    225  64.1
           2       No    126  35.9

In [8]:
# Target Distribution
target_df

,count,pct
GoodSignal,,
Yes,225,64.1
No,126,35.9


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to ionosphere/019d47b8-ec99-7c1c-91f5-20e67c05362e
019d47b8-ec99-7c1c-91f5-20e67c05362e
22d17c804df5b96d55f0f307f09e357bf19137b70d78ce7f23fb67242773c3b9
